# 01 - Comprensión del negocio y de los datos

Proyecto final de minería de datos orientado a la predicción de ventas mediante series de tiempo usando el Rossmann Store Sales Dataset.

## Objetivo general

Desarrollar un modelo predictivo de series de tiempo que permita estimar la demanda futura integrando variables externas para optimizar la planificación de marketing.

## Requerimientos del taller

**Técnicos:**

- Modelos ARIMA, Prophet y LSTM.
- Análisis de estacionalidad.
- Métricas RMSE, MAE y MAPE.

**Funcionales:**

- Dashboard de predicción.
- Simulación de escenarios.
- Visualización temporal.

## Fase CRISP-DM trabajada en este notebook

En este cuaderno se desarrolla la comprensión del negocio y la comprensión de los datos.  
Se cargan los archivos del dataset, se revisa su estructura y se integran las variables externas de las tiendas.

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", None)

import warnings
warnings.filterwarnings("ignore")

In [22]:
train = pd.read_csv("../01_data/train.csv")
test = pd.read_csv("../01_data/test.csv")
store = pd.read_csv("../01_data/store.csv")
sample_submission = pd.read_csv("../01_data/sample_submission.csv")

In [23]:
dimensiones = pd.DataFrame({
    "Archivo": ["train.csv", "test.csv", "store.csv", "sample_submission.csv"],
    "Filas": [train.shape[0], test.shape[0], store.shape[0], sample_submission.shape[0]],
    "Columnas": [train.shape[1], test.shape[1], store.shape[1], sample_submission.shape[1]]
})

dimensiones

,Archivo,Filas,Columnas
0,train.csv,1017209,9
1,test.csv,41088,8
2,store.csv,1115,10
3,sample_submission.csv,41088,2


## Exploración inicial

In [24]:
train.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


In [25]:
test.head()

,Id,Store,DayOfWeek,Date,Open,Promo,StateHoliday,SchoolHoliday
0,1,1,4,2015-09-17,1.00,1,0,0
1,2,3,4,2015-09-17,1.00,1,0,0
2,3,7,4,2015-09-17,1.00,1,0,0
3,4,8,4,2015-09-17,1.00,1,0,0
4,5,9,4,2015-09-17,1.00,1,0,0


In [26]:
test.head()

,Id,Store,DayOfWeek,Date,Open,Promo,StateHoliday,SchoolHoliday
0,1,1,4,2015-09-17,1.00,1,0,0
1,2,3,4,2015-09-17,1.00,1,0,0
2,3,7,4,2015-09-17,1.00,1,0,0
3,4,8,4,2015-09-17,1.00,1,0,0
4,5,9,4,2015-09-17,1.00,1,0,0


In [27]:
store.head()

,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,"1,270.00",9.00,"2,008.00",0,NaN,NaN,NaN
1,2,a,a,570.00,11.00,"2,007.00",1,13.00,"2,010.00","Jan,Apr,Jul,Oct"
2,3,a,a,"14,130.00",12.00,"2,006.00",1,14.00,"2,011.00","Jan,Apr,Jul,Oct"
3,4,c,c,620.00,9.00,"2,009.00",0,NaN,NaN,NaN
4,5,a,a,"29,910.00",4.00,"2,015.00",0,NaN,NaN,NaN


In [28]:
sample_submission.head()

,Id,Sales
0,1,0
1,2,0
2,3,0
3,4,0
4,5,0


In [29]:
columnas = pd.DataFrame({
    "train.csv": pd.Series(train.columns),
    "test.csv": pd.Series(test.columns),
    "store.csv": pd.Series(store.columns),
    "sample_submission.csv": pd.Series(sample_submission.columns)
})

columnas

,train.csv,test.csv,store.csv,sample_submission.csv
0,Store,Id,Store,Id
1,DayOfWeek,Store,StoreType,Sales
2,Date,DayOfWeek,Assortment,NaN
3,Sales,Date,CompetitionDistance,NaN
4,Customers,Open,CompetitionOpenSinceMonth,NaN
5,Open,Promo,CompetitionOpenSinceYear,NaN
6,Promo,StateHoliday,Promo2,NaN
7,StateHoliday,SchoolHoliday,Promo2SinceWeek,NaN
8,SchoolHoliday,NaN,Promo2SinceYear,NaN
9,NaN,NaN,PromoInterval,NaN


In [30]:
tipos_train = pd.DataFrame({
    "Columna": train.columns,
    "Tipo de dato": train.dtypes.astype(str).values,
    "Valores no nulos": train.notnull().sum().values,
    "Valores nulos": train.isnull().sum().values
})

tipos_train

,Columna,Tipo de dato,Valores no nulos,Valores nulos
0,Store,int64,1017209,0
1,DayOfWeek,int64,1017209,0
2,Date,object,1017209,0
3,Sales,int64,1017209,0
4,Customers,int64,1017209,0
5,Open,int64,1017209,0
6,Promo,int64,1017209,0
7,StateHoliday,object,1017209,0
8,SchoolHoliday,int64,1017209,0


In [31]:
tipos_test = pd.DataFrame({
    "Columna": test.columns,
    "Tipo de dato": test.dtypes.astype(str).values,
    "Valores no nulos": test.notnull().sum().values,
    "Valores nulos": test.isnull().sum().values
})

tipos_test

,Columna,Tipo de dato,Valores no nulos,Valores nulos
0,Id,int64,41088,0
1,Store,int64,41088,0
2,DayOfWeek,int64,41088,0
3,Date,object,41088,0
4,Open,float64,41077,11
5,Promo,int64,41088,0
6,StateHoliday,object,41088,0
7,SchoolHoliday,int64,41088,0


In [32]:
tipos_store = pd.DataFrame({
    "Columna": store.columns,
    "Tipo de dato": store.dtypes.astype(str).values,
    "Valores no nulos": store.notnull().sum().values,
    "Valores nulos": store.isnull().sum().values
})

tipos_store

,Columna,Tipo de dato,Valores no nulos,Valores nulos
0,Store,int64,1115,0
1,StoreType,object,1115,0
2,Assortment,object,1115,0
3,CompetitionDistance,float64,1112,3
4,CompetitionOpenSinceMonth,float64,761,354
5,CompetitionOpenSinceYear,float64,761,354
6,Promo2,int64,1115,0
7,Promo2SinceWeek,float64,571,544
8,Promo2SinceYear,float64,571,544
9,PromoInterval,object,571,544


In [33]:
nulos = pd.DataFrame({
    "Archivo": ["train.csv", "test.csv", "store.csv", "sample_submission.csv"],
    "Total valores nulos": [
        train.isnull().sum().sum(),
        test.isnull().sum().sum(),
        store.isnull().sum().sum(),
        sample_submission.isnull().sum().sum()
    ]
})

nulos

,Archivo,Total valores nulos
0,train.csv,0
1,test.csv,11
2,store.csv,2343
3,sample_submission.csv,0


In [34]:
nulos_store = pd.DataFrame({
    "Columna": store.columns,
    "Valores nulos": store.isnull().sum().values
})

nulos_store[nulos_store["Valores nulos"] > 0]

,Columna,Valores nulos
3,CompetitionDistance,3
4,CompetitionOpenSinceMonth,354
5,CompetitionOpenSinceYear,354
7,Promo2SinceWeek,544
8,Promo2SinceYear,544
9,PromoInterval,544


In [35]:
train.describe().T

,count,mean,std,min,25%,50%,75%,max
Store,"1,017,209.00",558.43,321.91,1.00,280.00,558.00,838.00,"1,115.00"
DayOfWeek,"1,017,209.00",4.00,2.00,1.00,2.00,4.00,6.00,7.00
Sales,"1,017,209.00","5,773.82","3,849.93",0.00,"3,727.00","5,744.00","7,856.00","41,551.00"
Customers,"1,017,209.00",633.15,464.41,0.00,405.00,609.00,837.00,"7,388.00"
Open,"1,017,209.00",0.83,0.38,0.00,1.00,1.00,1.00,1.00
Promo,"1,017,209.00",0.38,0.49,0.00,0.00,0.00,1.00,1.00
SchoolHoliday,"1,017,209.00",0.18,0.38,0.00,0.00,0.00,0.00,1.00


In [36]:
store.describe().T

,count,mean,std,min,25%,50%,75%,max
Store,"1,115.00",558.00,322.02,1.00,279.50,558.00,836.50,"1,115.00"
CompetitionDistance,"1,112.00","5,404.90","7,663.17",20.00,717.50,"2,325.00","6,882.50","75,860.00"
CompetitionOpenSinceMonth,761.00,7.22,3.21,1.00,4.00,8.00,10.00,12.00
CompetitionOpenSinceYear,761.00,"2,008.67",6.20,"1,900.00","2,006.00","2,010.00","2,013.00","2,015.00"
Promo2,"1,115.00",0.51,0.50,0.00,0.00,1.00,1.00,1.00
Promo2SinceWeek,571.00,23.60,14.14,1.00,13.00,22.00,37.00,50.00
Promo2SinceYear,571.00,"2,011.76",1.67,"2,009.00","2,011.00","2,012.00","2,013.00","2,015.00"


## Integración de variables externas

In [37]:
df = train.merge(store, on="Store", how="left")
df.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,"1,270.00",9.00,"2,008.00",0,NaN,NaN,NaN
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.00,11.00,"2,007.00",1,13.00,"2,010.00","Jan,Apr,Jul,Oct"
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,"14,130.00",12.00,"2,006.00",1,14.00,"2,011.00","Jan,Apr,Jul,Oct"
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.00,9.00,"2,009.00",0,NaN,NaN,NaN
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,"29,910.00",4.00,"2,015.00",0,NaN,NaN,NaN


In [38]:
dimension_df = pd.DataFrame({
    "Dataframe": ["train + store"],
    "Filas": [df.shape[0]],
    "Columnas": [df.shape[1]]
})

dimension_df

,Dataframe,Filas,Columnas
0,train + store,1017209,18


In [39]:
nulos_df = pd.DataFrame({
    "Columna": df.columns,
    "Valores nulos": df.isnull().sum().values
})

nulos_df[nulos_df["Valores nulos"] > 0]

,Columna,Valores nulos
11,CompetitionDistance,2642
12,CompetitionOpenSinceMonth,323348
13,CompetitionOpenSinceYear,323348
15,Promo2SinceWeek,508031
16,Promo2SinceYear,508031
17,PromoInterval,508031


In [40]:
variables_principales = [
    "Store",
    "Date",
    "Sales",
    "Customers",
    "Open",
    "Promo",
    "StateHoliday",
    "SchoolHoliday",
    "StoreType",
    "Assortment",
    "CompetitionDistance",
    "Promo2"
]

df[variables_principales].head()

,Store,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,Promo2
0,1,2015-07-31,5263,555,1,1,0,1,c,a,"1,270.00",0
1,2,2015-07-31,6064,625,1,1,0,1,a,a,570.00,1
2,3,2015-07-31,8314,821,1,1,0,1,a,a,"14,130.00",1
3,4,2015-07-31,13995,1498,1,1,0,1,c,c,620.00,0
4,5,2015-07-31,4822,559,1,1,0,1,a,a,"29,910.00",0


## Conclusión

Se cargaron los archivos del Rossmann Store Sales Dataset y se revisaron dimensiones, columnas, tipos de datos, valores nulos y estadísticas generales.

También se integraron los datos históricos de ventas con las variables externas de las tiendas mediante la columna `Store`.

La variable objetivo del proyecto es `Sales` y la variable temporal principal es `Date`.